In [ ]:
from transformers import pipeline
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

c:\Users\91787\Desktop\RAG_Sanskrit_abhishek_badawar\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
loader = Docx2txtLoader("Rag-docs.docx")
docs = loader.load()
docs

[Document(metadata={'source': 'Rag-docs.docx'}, page_content='मूर्खभृत्यस्य\n\n\n\n"अरे शंखनाद, गच्छापणम्, शर्कराम् आनय ।" इति स्वभृत्यम् शंखनादम् गोवर्धनदासः आदिशति । ततः शंखनादः आपणम् गच्छति, शर्कराम् जीर्णे वस्त्रे न्यस्यति च । तस्मात् जीर्णवस्त्रात् मार्गे एव सर्वापि शर्करा स्त्रवति । ततः गोवर्धनदासः कोपेन शंखनादम् वदति, "अरे मूढ, कुत्रास्ति शर्करा ? शर्करादिकम् एवम् जीर्णेन वस्त्रेण न एवानयन्ति कदापि । इतःपरम् किमपि वस्तुजातम् दृढायाम् सन्चिकायाम् निक्षिप्य आनय च " इति । अत्रान्तरे गोवर्धनदासस्य पुत्रः "श्वानशावकम् आनय" इति शंखनादम् आदिशति । आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च । तेन शावकस्य श्वासः रुध्दः भवति । सः च श्वानशावकः पञ्चत्वम् गच्छति । तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च, " धिक् मुढ, श्वानादिकम् दोरकेण बद्ध्वा आनयन्ति इति अपि नावगच्छसि किम् ?” इति । ततः कदाचित् गोवर्धनदासस्य भार्या "दुग्धम् आनय" इति तस्मै कथयति । सः आपणम् गच्छति पात्रे दुग्धम् आदाय दोरकेण बद्ध्वा कर्षति । मार्गे पात्रम् लुठति । पात्रात् दुग्ध

In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "।", "॥", " ", ""],
)

chunks = splitter.split_documents(docs)

In [14]:
chunks

[Document(metadata={'source': 'Rag-docs.docx'}, page_content='मूर्खभृत्यस्य'),
 Document(metadata={'source': 'Rag-docs.docx'}, page_content='"अरे शंखनाद, गच्छापणम्, शर्कराम् आनय ।" इति स्वभृत्यम् शंखनादम् गोवर्धनदासः आदिशति । ततः शंखनादः आपणम् गच्छति, शर्कराम् जीर्णे वस्त्रे न्यस्यति च । तस्मात् जीर्णवस्त्रात् मार्गे एव सर्वापि शर्करा स्त्रवति । ततः गोवर्धनदासः कोपेन शंखनादम् वदति, "अरे मूढ, कुत्रास्ति शर्करा ? शर्करादिकम् एवम् जीर्णेन वस्त्रेण न एवानयन्ति कदापि । इतःपरम् किमपि वस्तुजातम् दृढायाम् सन्चिकायाम् निक्षिप्य आनय च " इति । अत्रान्तरे गोवर्धनदासस्य पुत्रः "श्वानशावकम् आनय" इति शंखनादम् आदिशति'),
 Document(metadata={'source': 'Rag-docs.docx'}, page_content='। आज्ञापालकः शंखनादः श्वानशावकम् सन्चिकायाम् क्षिपति, सन्चिकाम् वस्त्रेण आच्छादयति च । तेन शावकस्य श्वासः रुध्दः भवति । सः च श्वानशावकः पञ्चत्वम् गच्छति । तदा गोवर्धनदासः शंखनादम् अभिधावति क्रोधेनाक्रोशति च, " धिक् मुढ, श्वानादिकम् दोरकेण बद्ध्वा आनयन्ति इति अपि नावगच्छसि किम् ?” इति । ततः कदाचित् गोवर्धनदासस्य भार्या "दुग्ध

In [15]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2272.22it/s]


In [17]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="sanskrit_db"
)

vectorstore

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [7]:
prompt = ChatPromptTemplate.from_template(
    """
You are a Sanskrit assistant.

Answer ONLY from the given context.

If answer is not available in context, say:
"Answer not found in the provided Sanskrit documents."

Context:
{context}

Question:
{input}

Answer:
"""
)

In [19]:
pipe = pipeline(
    task="text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    device=-1
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1037.75it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [20]:
rag_chain = (

    {
        "context": retriever,

        "input": RunnablePassthrough()
    }

    | prompt

    | llm

    | StrOutputParser()
)

In [21]:
response = rag_chain.invoke(
    "Why could no poet win the lakh rupees in King Bhoj’s court?"
)

In [25]:
response.split("Answer:")

['Human: \nYou are a Sanskrit assistant.\n\nAnswer ONLY from the given context.\n\nIf answer is not available in context, say:\n"Answer not found in the provided Sanskrit documents."\n\nContext:\n[Document(metadata={\'source\': \'Rag-docs.docx\'}, page_content=\'। न खलु वक्तुं अशक्नुवन् केऽपि विद्वानाः यत् जानन्ति तत् काव्यं इति । अतः अप्राप्नोत् कविः लक्षरुप्यकाणि । चतुरः खलु कालीदासः । (The poet read this poem in the court. None of the scholars were able to say that they knew this poetry. So the poet got one lakh rupees. kAlIdAsa was indeed clever!)\'), Document(metadata={\'source\': \'Rag-docs.docx\'}, page_content=\'। न खलु वक्तुं अशक्नुवन् केऽपि विद्वानाः यत् जानन्ति तत् काव्यं इति । अतः अप्राप्नोत् कविः लक्षरुप्यकाणि । चतुरः खलु कालीदासः । (The poet read this poem in the court. None of the scholars were able to say that they knew this poetry. So the poet got one lakh rupees. kAlIdAsa was indeed clever!)\'), Document(metadata={\'source\': \'Rag-docs.docx\'}, page_content=\'। न खलु

In [26]:
response = rag_chain.invoke(
    "Who was the servant in the story मूर्खभृत्यस्य?"
)


In [27]:
response.split("Answer:")

['Human: \nYou are a Sanskrit assistant.\n\nAnswer ONLY from the given context.\n\nIf answer is not available in context, say:\n"Answer not found in the provided Sanskrit documents."\n\nContext:\n[Document(metadata={\'source\': \'Rag-docs.docx\'}, page_content=\'वृद्धायाः चार्तुयम्\\n\\n\\n\\nआसीत् चित्रपुरम् नाम किमपि नगरं श्रीपर्वतस्य समीपे । "पर्वतस्य शिखरप्रदेशे घण्टाकर्णः नाम राक्षसः प्रतिवसती" ति जनप्रवादः अवर्तत् ।\\n\\nअथैकदा कश्चन चोरः घण्टामेकां चोरयित्वा वनं गतः, व्याघ्रण च हतः । तदा सा घण्टा वने एव अपतत् ।\\n\\nअन्यस्मिन् दिने केचन वानराः तत्र आगछन् । कुतुहलेन तां घण्टां हस्ते धृत्वा अधुन्वन् । अकस्मादेव घण्टानादः अजायत् ।\\n\\nघण्टानादेन चकिताः ते पुनः पुनः घण्टामधुन्वन्, घण्टाणादं च अकुर्वन् ।\'), Document(metadata={\'source\': \'Rag-docs.docx\'}, page_content=\'वृद्धायाः चार्तुयम्\\n\\n\\n\\nआसीत् चित्रपुरम् नाम किमपि नगरं श्रीपर्वतस्य समीपे । "पर्वतस्य शिखरप्रदेशे घण्टाकर्णः नाम राक्षसः प्रतिवसती" ति जनप्रवादः अवर्तत् ।\\n\\nअथैकदा कश्चन चोरः घण्टामेकां चोरयित्वा वनं गत